[◀ Notebook 08: Exception Handling](08_exception_handling.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 10: Magic Methods ▶](10_advanced_data_model.ipynb)

# Notebook 09: Classes and Object-Oriented Programming

Python is a multi-paradigm language that fully supports object-oriented programming (OOP). Unlike statically compiled languages, Python's class and type system is highly dynamic: classes are first-class objects, namespaces are dictionary-backed, and method binding occurs dynamically at runtime. This notebook explores the internal execution mechanics of Python classes, inheritance, Method Resolution Order (MRO), and attribute encapsulation.

---

## 1. Class and Instance Namespaces

In Python, namespaces are mapped to dictionaries under the hood. 

### How Class Definitions Execute
When Python encounters a `class` statement, it:
1. Creates a new dictionary namespace for the class body.
2. Executes the body of the class inside this namespace.
3. Creates a class object using the namespace dictionary (exposed via `__dict__`).

### Lookup Rules: Class vs. Instance Variables
- **Class Variables:** Declared in the class block. They are shared across all instances of the class.
- **Instance Variables:** Bound to `self` (typically within `__init__`). They are unique to each object instance.

When accessing an attribute via `obj.name`, Python performs a lookup hierarchy:
1. Search `obj.__dict__` (instance namespace).
2. Search `obj.__class__.__dict__` (class namespace).
3. Traverse the class's parent classes according to the Method Resolution Order (MRO).

> **Warning (Mutable Gotcha):** If a class-level variable points to a mutable object (like a list or dict), modifying it in-place via an instance (e.g., `self.list.append(x)`) alters the shared class variable. Reassigning it via an instance (e.g., `self.list = [x]`) masks the class variable by creating a new entry in `self.__dict__`.

In [ ]:
class Account:
    bank_name = "Pythonic Trust"  # Immutable class attribute
    transactions = []             # Mutable class attribute (shared!)

    def __init__(self, owner, balance):
        self.owner = owner        # Instance attribute
        self.balance = balance    # Instance attribute

# 1. Instantiation
alice = Account("Alice", 1000)
bob = Account("Bob", 500)

# 2. Inspecting the dictionaries
print("Alice's namespace:", alice.__dict__)
print("Account class namespace keys:", [k for k in Account.__dict__ if not k.startswith('__')])

# 3. Modifying a mutable class attribute affects all instances
alice.transactions.append("Alice deposited $50")
print("Bob's transactions:", bob.transactions)

# 4. Masking a class attribute by writing to the instance
alice.bank_name = "Global Bank"
print("Alice's bank name:", alice.bank_name)
print("Bob's bank name:", bob.bank_name)
print("Alice's namespace after masking:", alice.__dict__)

---

## 2. Object Life Cycle: `__new__` vs `__init__`

The instantiation process consists of two distinct stages:
1. **`__new__(cls, *args, **kwargs)` (Allocation):** A static method (implicitly treated as one) that allocates memory and returns a new, uninitialized instance of `cls`. It is the actual constructor.
2. **`__init__(self, *args, **kwargs)` (Initialization):** A method that receives the newly created instance as `self` and sets up initial attributes. It does not return anything (`None`).

Overriding `__new__` is essential for customizing the creation of immutable types (e.g., subclassing `tuple` or `str`) or when implementing architectural design patterns like the **Singleton Pattern**.

In [ ]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        print(f"[__new__] Allocating instance for {cls.__name__}")
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, val):
        print(f"[__init__] Initializing instance with value: {val}")
        self.val = val

s1 = Singleton(100)
s2 = Singleton(200)

print("s1 is s2:", s1 is s2)
print("s1.val:", s1.val)
print("s2.val:", s2.val)  # Note: __init__ runs again and overrides s1.val since it's the same object!

---

## 3. Method Binding Mechanics

Methods in Python classes are simply function objects defined within the class namespace. However, when accessed via an instance, they undergo **binding**:
- **Function (Unbound):** When accessed directly via the class (`Class.method`), it is a regular function object. Calling it requires passing an instance explicitly as the first argument.
- **Bound Method:** When accessed via an instance (`instance.method`), Python wraps the function in a bound method object. This object holds a reference to the instance and automatically inserts it as the first argument (`self`) when called.

In [ ]:
class Dog:
    def bark(self):
        return "Woof!"

d = Dog()

print("Dog.bark type:", type(Dog.bark))   # function
print("d.bark type:", type(d.bark))       # method

# Invocation comparison
print("Calling bound method:", d.bark())
print("Calling unbound function:", Dog.bark(d))

---

## 4. Inheritance & C3 Linearization (MRO)

Python supports multiple inheritance. When resolving method calls, Python must flatten the inheritance graph into a linear search path. Since Python 2.3 (PEP 253), this search path is calculated using **C3 Linearization** to define the **Method Resolution Order (MRO)**.

### Rules of C3 Linearization
1. **Local Precedence Order:** Children are resolved before parents. If `class A(B, C)` is declared, `B` is checked before `C`.
2. **Monotonicity:** If a class precedes another in a parent's MRO, it must preserve that relationship in all derived MROs.
3. **Single Inheritance Consistency:** If a class inherits only from single classes, the MRO is simply grandparent, parent, child.

### Cooperative Multiple Inheritance with `super()`
The built-in function `super()` does not simply call the parent class's method. Instead, it delegates to the next class in the **current object's MRO**. This enables cooperative multiple inheritance, where sibling classes can chain method execution using cooperative `super()` calls.

In [ ]:
class Base:
    def greet(self):
        print(" Base greet")

class A(Base):
    def greet(self):
        print(" A starting")
        super().greet()
        print(" A ending")

class B(Base):
    def greet(self):
        print(" B starting")
        super().greet()
        print(" B ending")

class Derived(A, B):
    def greet(self):
        print(" Derived starting")
        super().greet()
        print(" Derived ending")

# Display MRO
print("MRO of Derived:", [cls.__name__ for cls in Derived.__mro__])

# Run and trace the cooperative calls
d = Derived()
d.greet()

---

## 5. Method Classifications: Instance, Class, and Static

Python supports three primary decorators to specify method behavior:
1. **Instance Methods (Default):** Receive `self` as the first argument. Can access/modify instance-level state and class-level state.
2. **Class Methods (`@classmethod`):** Receive the class object (`cls`) as the first argument. Useful for defining factory methods (alternative constructors).
3. **Static Methods (`@staticmethod`):** Receive no automatic first argument. They act like regular functions residing inside the class namespace, useful for isolation of stateless utility logic.

In [ ]:
class TempConverter:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        # Alternative constructor
        celsius = (fahrenheit - 32) * 5 / 9
        return cls(celsius)

    @staticmethod
    def validate_temp(celsius):
        # Statelessly check boundaries
        return celsius >= -273.15

temp = TempConverter.from_fahrenheit(98.6)
print(f"Celsius temp: {temp.celsius:.2f}")
print("Absolute zero validation:", TempConverter.validate_temp(-300))

---

## 6. Encapsulation: Properties & Name Mangling

- **Naming Conventions:** A single leading underscore (e.g., `_attribute`) signifies an internal/private implementation detail by convention.
- **Name Mangling:** A double leading underscore (e.g., `__attribute`) forces the Python interpreter to rewrite the attribute name internally as `_ClassName__attribute`. This prevents accidental collisions in subclass hierarchies.
- **Properties (`@property`):** Allows getters, setters, and deleters to be accessed using clean dot-notation access while enforcing data validation rules under the hood.

---

## 7. Dynamic Attribute Manipulation & Reflection

Python is an exceptionally dynamic language where attribute bindings can be inspected, added, modified, or removed at runtime. This capability, known as **reflection** or **introspection**, allows developers to write highly flexible, meta-programmed APIs (such as serialization libraries, dependency injection containers, or routing tables).

Python's reflection API is built around four built-in functions:
- **`hasattr(object, name)`**: Checks if the object has an attribute with the given name (passed as a string). Under the hood, CPython implements this by calling `getattr(object, name)` and returning `False` if an `AttributeError` is raised.
- **`getattr(object, name[, default])`**: Retrieves the value of the named attribute. If the attribute does not exist, it returns the provided `default` value, or raises an `AttributeError` if no default is specified.
- **`setattr(object, name, value)`**: Assigns the value to the named attribute on the object, creating the attribute dynamically if it does not already exist.
- **`delattr(object, name)`**: Deletes the named attribute from the object's namespace.

### CPython Implementation & Namespace Mechanics
For standard user-defined objects, attributes are stored in a dictionary exposed as `__dict__`. When reflection functions are used, CPython executes specific bytecode or API calls:
1. `getattr(obj, 'x')` invokes `type(obj).__getattribute__(obj, 'x')`. This traverses the class MRO to check for data descriptors, then checks the instance `__dict__`, then checks non-data descriptors, and finally falls back to `__getattr__` if it exists.
2. `setattr(obj, 'x', val)` checks the MRO for a data descriptor with a `__set__` method. If none exists, it writes the value to `obj.__dict__['x']`.
3. These operations bypass compile-time checks entirely, enabling late binding where attribute names are determined dynamically at runtime (e.g., parsed from configuration or incoming JSON payloads).

### Dynamic Attribute Manipulation in Practice:
```python
class DynamicRecord:
    def __init__(self):
        self.status = "active"
    def run(self):
        return "Running task"

record = DynamicRecord()

# Check and get attributes dynamically
attribute_name = "status"
if hasattr(record, attribute_name):
    value = getattr(record, attribute_name)
    print(f"{attribute_name} value: {value}")

# Set attributes dynamically
setattr(record, "priority", "high")
print(f"priority value: {record.priority}")

# Execute methods dynamically
method_name = "run"
if hasattr(record, method_name):
    func = getattr(record, method_name)
    print(f"Method execution: {func()}")

# Delete attributes dynamically
delattr(record, "priority")
print("Has priority?:", hasattr(record, "priority"))
```

In [ ]:
class SecuredVault:
    def __init__(self, key):
        self.__key = key  # Name-mangled variable

    @property
    def key(self):
        return "*******"

    @key.setter
    def key(self, value):
        if not isinstance(value, str):
            raise TypeError("Vault key must be a string!")
        self.__key = value

    @key.deleter
    def key(self):
        print("Vault key deleted!")
        del self.__key

vault = SecuredVault("my_secret_key")
print("Reading key property:", vault.key)
vault.key = "new_secret_key"
del vault.key

# Dynamic reflection attribute manipulation
print("\nDynamic Reflection:")
print("hasattr(vault, 'key'):", hasattr(vault, 'key'))
setattr(vault, 'vault_location', 'Zurich')
print("vault_location attribute via getattr:", getattr(vault, 'vault_location'))
delattr(vault, 'vault_location')
print("hasattr after delattr:", hasattr(vault, 'vault_location'))


---

## Coding Challenges

Complete the following exercises to test your understanding of classes and object-oriented programming in Python. Run the assertion tests to verify correctness.

### Challenge 1: Cooperative Multiple Inheritance & MRO

Implement an order-pricing system consisting of four classes:
1. `BaseOrder`: Implements a method `compute_price(self, amount)` that returns the `amount` unchanged.
2. `DiscountMixin`: Subclasses `BaseOrder`. Overrides `compute_price` to apply a 10% discount to the calculated value of `super().compute_price(amount)` (i.e. returning `0.90 * super().compute_price(amount)`).
3. `TaxMixin`: Subclasses `BaseOrder`. Overrides `compute_price` to apply a 5% tax to the calculated value of `super().compute_price(amount)` (i.e. returning `1.05 * super().compute_price(amount)`).
4. `FinalOrder`: Subclasses both `DiscountMixin` and `TaxMixin` cooperatives. It should not override `compute_price` directly.

In [ ]:
class BaseOrder:
    def compute_price(self, amount):
        # TODO: Implement BaseOrder
        return amount

class DiscountMixin(BaseOrder):
    def compute_price(self, amount):
        # TODO: Implement DiscountMixin using super()
        return 0.90 * super().compute_price(amount)

class TaxMixin(BaseOrder):
    def compute_price(self, amount):
        # TODO: Implement TaxMixin using super()
        return 1.05 * super().compute_price(amount)

class FinalOrder(DiscountMixin, TaxMixin):
    # TODO: Inherit from both mixins to establish correct MRO
    pass

In [ ]:
# Test Challenge 1
order = FinalOrder()

# Verify correct C3 Linearization order
mro_names = [cls.__name__ for cls in FinalOrder.__mro__]
print("FinalOrder MRO:", mro_names)
assert mro_names[:5] == ['FinalOrder', 'DiscountMixin', 'TaxMixin', 'BaseOrder', 'object'], "Incorrect MRO hierarchy!"

# Verify cooperative calculation (100 -> tax 5% -> discount 10% -> 94.5)
price = order.compute_price(100.0)
print(f"Computed Price: {price}")
assert abs(price - 94.5) < 1e-9, f"Expected 94.5, but got {price}"
print("Challenge 1 passed!")

### Challenge 2: Input-Validating Attribute Class Wrapper

Write a class `Product` that manages product attributes with properties:
1. `name`: Must be a non-empty string. Raise `TypeError` if not a string. Raise `ValueError` if it is an empty string.
2. `price`: Must be an `int` or `float` greater than or equal to `0.0`. Raise `TypeError` if not a number. Raise `ValueError` if negative.
3. `sku` (Read-only): Should automatically return `f"PROD-{self.name.upper()}"`. It must update dynamically if the product `name` changes. No setter should exist.

In [ ]:
class Product:
    def __init__(self, name: str, price: float):
        # TODO: Initialize attributes through properties
        self.name = name
        self.price = price

    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str):
        # TODO: Implement name setter validation
        if not isinstance(value, str):
            raise TypeError("Name must be a string")
        if not value.strip():
            raise ValueError("Name cannot be empty")
        self._name = value

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float):
        # TODO: Implement price setter validation
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise TypeError("Price must be a number")
        if value < 0.0:
            raise ValueError("Price cannot be negative")
        self._price = float(value)

    @property
    def sku(self) -> str:
        # TODO: Implement sku getter
        return f"PROD-{self.name.upper()}"

In [ ]:
# Test Challenge 2
p = Product("Laptop", 999.99)
assert p.name == "Laptop"
assert p.price == 999.99
assert p.sku == "PROD-LAPTOP"

# Verify dynamically updated sku
p.name = "Tablet"
assert p.sku == "PROD-TABLET"

# Verify Name Validations
try:
    p.name = ""
    assert False, "Allowed empty name"
except ValueError:
    pass

try:
    p.name = 123
    assert False, "Allowed integer name"
except TypeError:
    pass

# Verify Price Validations
try:
    p.price = -10.0
    assert False, "Allowed negative price"
except ValueError:
    pass

try:
    p.price = "one hundred"
    assert False, "Allowed string price"
except TypeError:
    pass

# Verify sku is read-only
try:
    p.sku = "PROD-NEW"
    assert False, "sku is writable"
except AttributeError:
    pass

print("Challenge 2 passed!")

### Challenge 3: A Class Instance Registry

Implement a class `RegisteredModel` that monitors all active instances partitioned by category.
1. Define a class attribute dictionary named `registry`.
2. In `__init__(self, name: str, category: str)`, save name and category, and add the object to `registry` under the matching category key mapped to a list of instances.
3. Implement a class method `get_by_category(cls, category: str)` that returns a list of all instantiated objects in that category.
4. Implement a class method `clear_registry(cls)` that removes all entries in the registry.

In [ ]:
class RegisteredModel:
    # TODO: Define the registry class variable
    registry = {}

    def __init__(self, name: str, category: str):
        # TODO: Initialize name and category, and register the instance
        self.name = name
        self.category = category
        if category not in self.registry:
            self.registry[category] = []
        self.registry[category].append(self)

    @classmethod
    def get_by_category(cls, category: str):
        # TODO: Return matching instances or empty list
        return cls.registry.get(category, [])

    @classmethod
    def clear_registry(cls):
        # TODO: Clear the registry dictionary
        cls.registry.clear()

In [ ]:
# Test Challenge 3
RegisteredModel.clear_registry()

m1 = RegisteredModel("Linear Regression", "regression")
m2 = RegisteredModel("Random Forest", "classification")
m3 = RegisteredModel("Ridge Regression", "regression")

reg_models = RegisteredModel.get_by_category("regression")
clf_models = RegisteredModel.get_by_category("classification")

assert len(reg_models) == 2
assert len(clf_models) == 1
assert m1 in reg_models
assert m3 in reg_models
assert m2 in clf_models

RegisteredModel.clear_registry()
assert len(RegisteredModel.get_by_category("regression")) == 0

print("Challenge 3 passed!")

[◀ Notebook 08: Exception Handling](08_exception_handling.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 10: Magic Methods ▶](10_advanced_data_model.ipynb)